In [9]:

import cv2
import numpy as np
from collections import deque
from ultralytics import YOLO
from IPython.display import Video

VIDEO_PATH="/content/volleyball_match.mp4"
OUTPUT_PATH="/content/output_volleyball.mp4"

# High resolution for better ball stability
WIDTH,HEIGHT = 960,540

trail=deque(maxlen=30)
hist_top=[]
hist_bottom=[]

model=YOLO("yolov8n.pt")

court_roi=np.array([[180,105],[780,105],[930,495],[30,495]],
                   np.int32).reshape((-1,1,2))

bg=cv2.createBackgroundSubtractorMOG2(300,16,False)

# Tracking state
prev_ball=None
velocity=np.array([0.0,0.0])
lost=0
MAX_LOST=8


def get_team(frame,box):
    x1,y1,x2,y2=box
    torso=frame[y1:y1+int((y2-y1)*0.4),x1:x2]
    if torso.size==0:
        return None
    hsv=cv2.cvtColor(torso,cv2.COLOR_BGR2HSV)
    red=cv2.inRange(hsv,(170,180,140),(180,255,255))
    yellow=cv2.inRange(hsv,(8,160,150),(30,255,255))
    if np.sum(red)>np.sum(yellow) and np.sum(red)>50:
        return "red"
    if np.sum(yellow)>np.sum(red) and np.sum(yellow)>50:
        return "yellow"
    return None


cap=cv2.VideoCapture(VIDEO_PATH)
fps=cap.get(cv2.CAP_PROP_FPS) or 30

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (WIDTH, HEIGHT)
)


while True:
    ret,frame=cap.read()
    if not ret:
        break

    frame=cv2.resize(frame,(WIDTH,HEIGHT))
    clean=frame.copy()
    hsv=cv2.cvtColor(clean,cv2.COLOR_BGR2HSV)
    mid=HEIGHT//2

    #PLAYERS
    results=model(clean,verbose=False)[0]
    player_boxes=[]
    top=0
    bottom=0

    for box,cls in zip(results.boxes.xyxy,results.boxes.cls):
        if int(cls)!=0:
            continue

        x1,y1,x2,y2=map(int,box)
        if (y2-y1)<60:
            continue

        feet=((x1+x2)//2,y2)
        if cv2.pointPolygonTest(court_roi,feet,False)<0:
            continue

        team=get_team(clean,(x1,y1,x2,y2))
        if team is None:
            continue

        player_boxes.append((x1,y1,x2,y2))

        cy=(y1+y2)//2
        if cy<mid:
            top+=1
        else:
            bottom+=1

        cv2.rectangle(frame,(x1,y1),(x2,y2),
                      (0,255,0) if cy<mid else (0,0,255),2)

    #BALL SEARCH
    search=clean.copy()
    for x1,y1,x2,y2 in player_boxes:
        cv2.rectangle(search,(x1,y1),(x2,y2),(0,0,0),-1)

    blur=cv2.GaussianBlur(search,(11,11),0)
    hsv_ball=cv2.cvtColor(blur,cv2.COLOR_BGR2HSV)

    yellow=cv2.inRange(hsv_ball,(10,40,80),(40,255,255))
    blue=cv2.inRange(hsv_ball,(90,50,50),(130,255,255))
    white=cv2.inRange(hsv_ball,(0,0,200),(180,60,255))

    color=cv2.bitwise_or(yellow,blue)
    color=cv2.bitwise_or(color,white)

    motion=bg.apply(blur)
    motion=cv2.morphologyEx(motion,cv2.MORPH_OPEN,np.ones((5,5),np.uint8))

    mask=cv2.bitwise_and(color,motion)
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((7,7),np.uint8))

    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

    candidate=None
    best_score=1e9

    for cnt in contours:
        area=cv2.contourArea(cnt)
        if area<40 or area>1500:
            continue

        peri=cv2.arcLength(cnt,True)
        if peri==0:
            continue

        circ=4*np.pi*area/(peri*peri)
        if circ<0.6:
            continue

        x,y,w,h=cv2.boundingRect(cnt)
        aspect=w/float(h)
        if not (0.8<aspect<1.25):
            continue

        cx=x+w//2
        cy=y+h//2

        score=(1-circ)*80

        # Physics gating
        if prev_ball is not None:
            pred=np.array(prev_ball)+velocity
            dist=np.linalg.norm(pred-np.array([cx,cy]))
            if dist>220:
                continue
            score+=dist

        if score<best_score:
            best_score=score
            candidate=(cx,cy)

    #track
    if candidate is not None:
        if prev_ball is not None:
            velocity=0.7*velocity + 0.3*(np.array(candidate)-np.array(prev_ball))
        prev_ball=candidate
        trail.appendleft(candidate)
        lost=0
    else:
        lost+=1
        if lost>MAX_LOST:
            prev_ball=None
            velocity=np.array([0.0,0.0])
            trail.appendleft(None)

    # draw ball marker
    if prev_ball is not None:
        cv2.circle(frame, prev_ball, 6, (0,255,255), -1)
        cv2.circle(frame, prev_ball, 12, (0,255,255), 2)

    #trail
    for i in range(1,len(trail)):
        if trail[i] is None or trail[i-1] is None:
            continue
        d=np.linalg.norm(np.array(trail[i])-np.array(trail[i-1]))
        if d>160:
            continue
        thickness=int(4*(1-i/len(trail)))+1
        cv2.line(frame,trail[i-1],trail[i],(0,255,255),thickness)

    #hud
    hist_top.append(top)
    hist_bottom.append(bottom)
    top=int(np.median(hist_top[-6:]))
    bottom=int(np.median(hist_bottom[-6:]))

    cv2.putText(frame,f"Top:{top}",(10,30),
                cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,255,0),2)
    cv2.putText(frame,f"Bottom:{bottom}",(10,65),
                cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,0,255),2)

    out.write(frame)

cap.release()
out.release()
!ffmpeg -y -i /content/output_volleyball.mp4 -vcodec libx264 /content/final_output.mp4


from IPython.display import Video
Video("/content/final_output.mp4", embed=True)

from google.colab import files
files.download("/content/final_output.mp4")



ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.7 MB/s eta 0:00:00
